# 4-ball carom env smoke test

Round-trip the Gymnasium environment + the inline HTML replay viewer for three representative shots:

1. **clean** — straight at red-1, no spin
2. **cushion-rich** — 30 ° off-axis, moderate power
3. **draw + side spin** — power 0.7, b = -0.95 (draw), a = 0.4 (right english, clipped to the unit disk)

Each shot prints a one-line summary, renders the trajectory, and the final cell shows a comparison table.

In [ ]:
import math

import gymnasium  # noqa: F401  (registry warm-up; we use Billiards4BallEnv directly)
import numpy as np
from IPython.display import HTML

In [ ]:
from billiards import Billiards4BallEnv
from billiards.render.replay import render_html

In [ ]:
env = Billiards4BallEnv()
obs, info = env.reset(seed=42)
print('obs shape:', obs.shape)
print('table spec:', info['spec'])

## Shot 1 — clean: straight at red-1, no spin

Compute `theta` from the initial cue / red-1 layout so the cue ball is aimed at the foot spot.

In [ ]:
obs, info = env.reset(seed=42)
init = obs.reshape(4, 7)
cue_xy = init[0, :2]
red1_xy = init[2, :2]
theta1 = math.atan2(red1_xy[1] - cue_xy[1], red1_xy[0] - cue_xy[0])
action1 = np.array([theta1, 0.6, 0.0, 0.0], dtype=np.float32)
_, reward1, term, trunc, info1 = env.step(action1)
print(f"shot 1: theta={theta1:.3f} rad  power=0.6  spin=(0,0)")
print(
    f"  score={info1['score']}  fouled={info1['fouled']}"
    f"  cushion_hits={info1['cushion_hits']}"
    f"  duration={info1['duration']:.3f}s  events={len(info1['event_log'])}"
)
print('  terminated/truncated:', term, trunc)

In [ ]:
html1 = render_html(info1['trajectory'], spec=info1['spec'])
html1

## Shot 2 — cushion-rich: 30 ° off-axis, power 0.55

Aim 30 ° above the head→foot baseline so the cue ball is forced onto a long-rail. No spin — the deflection has to come from cushion bounces.

In [ ]:
obs, info = env.reset(seed=42)
init = obs.reshape(4, 7)
cue_xy = init[0, :2]
red1_xy = init[2, :2]
base = math.atan2(red1_xy[1] - cue_xy[1], red1_xy[0] - cue_xy[0])
theta2 = base + math.radians(30.0)
action2 = np.array([theta2, 0.55, 0.0, 0.0], dtype=np.float32)
_, reward2, term, trunc, info2 = env.step(action2)
print(f"shot 2: theta={theta2:.3f} rad (base+30°)  power=0.55  spin=(0,0)")
print(
    f"  score={info2['score']}  fouled={info2['fouled']}"
    f"  cushion_hits={info2['cushion_hits']}"
    f"  duration={info2['duration']:.3f}s  events={len(info2['event_log'])}"
)

In [ ]:
render_html(info2['trajectory'], spec=info2['spec'])

## Shot 3 — draw + side spin

`b = -0.95` is heavy draw (back-spin); `a = 0.4` is right english. The combination `(0.4, -0.95)` lies just outside the unit disk (`r = 1.0307 > 1`) and is radially projected back onto the disk by the env's action wrapper. We log the projected `(a, b)` so the notebook surfaces the clip.

In [ ]:
obs, info = env.reset(seed=42)
init = obs.reshape(4, 7)
cue_xy = init[0, :2]
red1_xy = init[2, :2]
theta3 = math.atan2(red1_xy[1] - cue_xy[1], red1_xy[0] - cue_xy[0])
raw_a, raw_b = 0.4, -0.95
r = math.hypot(raw_a, raw_b)
if r > 1.0:
    proj_a, proj_b = raw_a / r, raw_b / r
else:
    proj_a, proj_b = raw_a, raw_b
action3 = np.array([theta3, 0.7, raw_a, raw_b], dtype=np.float32)
_, reward3, term, trunc, info3 = env.step(action3)
print(
    f"shot 3: theta={theta3:.3f} rad  power=0.7  raw spin=({raw_a},{raw_b})"
    f" -> projected=({proj_a:.3f},{proj_b:.3f})"
)
print(
    f"  score={info3['score']}  fouled={info3['fouled']}"
    f"  cushion_hits={info3['cushion_hits']}"
    f"  duration={info3['duration']:.3f}s  events={len(info3['event_log'])}"
)

In [ ]:
render_html(info3['trajectory'], spec=info3['spec'])

## Summary

In [ ]:
rows = [
    ('shot 1 (clean)',          info1),
    ('shot 2 (cushion-rich)',   info2),
    ('shot 3 (draw+side spin)', info3),
]
header = f"{'shot':<24} {'score':>5} {'fouled':>6} {'cush':>4} {'dur(s)':>7} {'events':>6}"
print(header)
print('-' * len(header))
for name, inf in rows:
    print(
        f"{name:<24} {inf['score']:>5} {str(inf['fouled']):>6}"
        f" {inf['cushion_hits']:>4} {inf['duration']:>7.3f}"
        f" {len(inf['event_log']):>6}"
    )